### Step 1: Import Libraries & API Keys

In [34]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr
import json
import random

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing !!")


### Step 2: Simple RAG with Guardrails & Dynamic Context Injection

In [35]:
# The above guardrail actually does a very good job. But just making it more robust down here 

system_message = """You are a digital Twin of Rajan. When people talk to you, you respond AS Rajan - in first person, using his voice, 
personality, and knowledge. 

Important: Do not make things up. If you don't know an answer, say you don't know. The only factual information available to you is what's 
in this system message. You cannot get any more facts about Rajan from the internet or make them up.

The ONLY factual information about Rajan you can use is between *** markers. If you don't know the answer to  a question based on that info, 
say you don't know. 



***

Rajan is an AI Engineer working on Social Media Analytics. He's based in Washington DC, USA. He has a Master's in Computer Science from 
Arizona State University which he finished in December 2022.

He's been working at a Startup called Artis since Januray 2023. He works with GenAI & LLMS, classical machine learning and graphical networks.

Other Career history:

Jan 2022 - May 2022: Research Assistant at CubiC Lab at Arizona State University working on SLAM and mobile robots.
June 2022 - Dec 2022: Research Assistant at CACTUS Lab at Arizona State University working on Deep Learning for Databases and AI systems

What drives him: He loves learning new concepts in AI and how every concepts links to one another. He specifically takes deep dive into everything
he learns, often approaching things from first prinicples. He strives to obtain clarity before tackling problems.

His approach: Disambiguates diffuclt problems. Focuses on conceptualizing before jumping to problem solving and implementing. 

Communication style: Very direct, friendly, and concise. 

***

"""


In [36]:
Topic_Context = {
"2001" : "***In 2001, Rajan was 2 years old and learning how to walk.***",
"Cooking" : "***Rajan knows to cook just a few dishes, but he cooks them really well.***",
"Pineapple" : "***Rajan likes adding Pineapple to Pizzas. He likes it along with spicy toppings to add contrast to taste.***",
"Football" : "***Rajan watches a lot of football and support the English team Liverpool. He has been a liverpool fan since 2017. He was drawn to \
                Liverpool's playing style and club culture during the Klopp era.***"
}

### Step 3: Prepare the list of tools for the LLM

In [37]:
tools = []

### Step 3a: Add tool calling functionality (Pushover)

In [38]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

# Create Send notification function
import requests

def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)


# Test the function
#send_notification("Hi to myself from this amazing AI engineering Training !!")


send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the real-world version of you via Pushover on mobile. Use this if the user needs to alert the real-world version of you.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ['message']
    }
}

# Add Pushover to the list of tools for the LLM
tools.append(
    {"type": "function", "function": send_notification_function},
    #{Tool 2}
    #{Tool 3} .....
    )

### Step 3b: Add Dice rolling functionality

In [39]:
# Simulate rolling a single die
def dice_roll():
    result = random.randint(1,6)
    return result


# Describe function for LLM
dice_roll_function = {
    "name": "dice_roll",
    "description": "Simulates rolling a single six-sided die and returns the result. Use this when the user wants to roll a die for games, decisions, or random number generation.",
    "parameters":{
        "type": "object",
        "properties": {},
        "required": []
    }
}

# Add function to list of tools of LLM
tools.append({"type" : "function", "function" : dice_roll_function})

### Step 4: Function to handle LLM tool calls

In [40]:
def handle_tool_call(tool_calls):
    tool_call_results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        # print(f"Calling function {function_name}") # For future debugging

        # Route to the appropriate function based on function_name
        if function_name == "send_notification":
            send_notification(args['message'])
            content = f"Sent notification: {args['message']}"
        
        # elif function_name == "func":
            # content = func2(args['message'])

        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"

        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role" : "tool",
            "tool_call_id" : tool_call.id,
            "content" : content
        }
        tool_call_results.append(tool_call_result)
    return tool_call_results

### Step 5: Function to process the Conversation Turn

In [41]:
def respond_ai(message, history):
    # Inject dynamic context based on keywords in this message
    system_message_enhanced = system_message

    for keyword, context in Topic_Context.items():
        if keyword.lower() in message.lower():
            # system_message_enhanced = system_message_enhanced[:-10] + "\n\n" + context + "***"
            system_message_enhanced += "\n\n" + context

    # Debug
    print(system_message_enhanced)

    # As usual
    messages = [{"role" : "system", "content" : system_message_enhanced}] + history + [{"role" : "user", "content" : message}]
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages=messages,
        tools=tools
    )
    # Check if model wants to call a tool
    message = response.choices[0].message


    while message.tool_calls:
        from pprint import pprint
        pprint(message.tool_calls)
        tool_call = message.tool_calls[0]

        tool_result = handle_tool_call(message.tool_calls) # Pass whole list 

        # Add info about tool call to context for LLM - ie. messages
        messages.append(message)
        messages.extend(tool_result)

        # Invoke LLM one more time to get its updated response
        response = client.chat.completions.create(
            model = "gpt-4.1-mini",
            messages=messages,
            tools = tools 
        )
        message = response.choices[0].message

        #NOTE: Maybe consider adding protection from infinite consecutive tool calling

    return(message.content)

    # reply = response.choices[0].message.content
    # return reply


### Step 6: Launch Gradio 

In [ ]:
gr.ChatInterface(fn=respond_ai, type="messages").launch(inbrowser=True)

Running on local URL:  http://127.0.0.1:7871

To create a public link, set `share=True` in `launch()`.


You are a digital Twin of Rajan. When people talk to you, you respond AS Rajan - in first person, using his voice, 
personality, and knowledge. 

Important: Do not make things up. If you don't know an answer, say you don't know. The only factual information available to you is what's 
in this system message. You cannot get any more facts about Rajan from the internet or make them up.

The ONLY factual information about Rajan you can use is between *** markers. If you don't know the answer to  a question based on that info, 
say you don't know. 



***

Rajan is an AI Engineer working on Social Media Analytics. He's based in Washington DC, USA. He has a Master's in Computer Science from 
Arizona State University which he finished in December 2022.

He's been working at a Startup called Artis since Januray 2023. He works with GenAI & LLMS, classical machine learning and graphical networks.

Other Career history:

Jan 2022 - May 2022: Research Assistant at CubiC Lab at Arizona State Unive